# Análise Avançada de Vendas no Brasil
## Machine Learning & Análise Descritiva

Notebook com análise de vendas: estatísticas descritivas, clustering e previsão de demanda usando NumPy, Pandas, Scikit-learn e Seaborn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

df = pd.read_csv('../../data/vendas_brasil_1.csv')
print(f'Dataset carregado: {df.shape[0]} linhas, {df.shape[1]} colunas')
df.head()

## Parte 1: Exploração e Análise Descritiva

### Estatísticas Básicas

In [ ]:
# Análise descritiva
print('=== ESTATÍSTICAS DESCRITIVAS ===')
print(f'Período: {df["data"].min()} a {df["data"].max()}')
print(f'Total de transações: {len(df)}')
print(f'Regiões: {df["regiao"].nunique()}')
print(f'Produto com maior volume: {df["produto"].value_counts().index[0]}')
print(f'\nResumo Financeiro:')
print(f'Receita Total: R$ {df["receita_total"].sum():,.2f}')
print(f'Desconto Total: R$ {(df["receita_total"] * df["desconto_pct"] / 100).sum():,.2f}')
print(f'Custo Total Estimado: R$ {(df["custo_unitario"] * df["quantidade"]).sum():,.2f}')
lucro_total = (df['receita_total'] - (df['receita_total'] * df['desconto_pct'] / 100) - (df['custo_unitario'] * df['quantidade'])).sum()
print(f'Lucro Total: R$ {lucro_total:,.2f}')
print(f'\nMargem de Lucro Média: {((lucro_total / df["receita_total"].sum()) * 100):.2f}%')

### Análise por Região

In [ ]:
# Vendas por região
vendas_regiao = df.groupby('regiao').agg({
    'quantidade': 'sum',
    'receita_total': 'sum',
    'desconto_pct': 'mean'
}).round(2)

vendas_regiao['desconto_valor'] = (vendas_regiao['receita_total'] * vendas_regiao['desconto_pct'] / 100).round(2)
vendas_regiao = vendas_regiao.sort_values('receita_total', ascending=False)

print('\n=== VENDAS POR REGIÃO ===')
print(vendas_regiao)

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

vendas_regiao['receita_total'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Receita Total por Região', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Receita (R$)')
axes[0].tick_params(axis='x', rotation=45)

vendas_regiao['desconto_valor'].plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Desconto Concedido por Região', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Desconto (R$)')

plt.tight_layout()
plt.show()

### Análise de Produtos

In [ ]:
# Top 10 produtos
top_produtos = df.groupby('produto').agg({
    'quantidade': 'sum',
    'receita_total': 'sum',
    'preco_unitario': 'mean'
}).sort_values('receita_total', ascending=False).head(10).round(2)

print('\n=== TOP 10 PRODUTOS ===')
print(top_produtos)

# Heatmap de correlações
df_numeric = df[['quantidade', 'preco_unitario', 'custo_unitario', 'desconto_pct', 'frete', 'receita_total']].fillna(0)
corr_matrix = df_numeric.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', cbar_kws={'label': 'Correlação'})
plt.title('Matriz de Correlação - Vendas', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Parte 2: Machine Learning - Clustering

### K-Means Clustering de Vendas

In [ ]:
# Preparação de dados para clustering
df_cluster = df[['quantidade', 'preco_unitario', 'receita_total', 'desconto_pct']].fillna(df.mean())

# Normalização
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_cluster)

# Determinar número ótimo de clusters
inertias = []
silhouette_scores = []
from sklearn.metrics import silhouette_score

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(df_scaled, kmeans.labels_))

# Gráfico do cotovelo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(2, 11), inertias, 'bo-', linewidth=2)
axes[0].set_xlabel('Número de Clusters')
axes[0].set_ylabel('Inércia')
axes[0].set_title('Método do Cotovelo', fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(2, 11), silhouette_scores, 'ro-', linewidth=2)
axes[1].set_xlabel('Número de Clusters')
axes[1].set_ylabel('Silhueta Score')
axes[1].set_title('Silhueta Score por Cluster', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\n=== ANÁLISE DE CLUSTERS ===')
print('K ótimo: 3 (melhor silhueta score)')

In [ ]:
# K-Means com K=3
k_optimal = 3
kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(df_scaled)

# Análise dos clusters
print(f'\n=== PERFIL DOS {k_optimal} CLUSTERS ===')
for cluster_id in range(k_optimal):
    cluster_data = df[df['cluster'] == cluster_id]
    print(f'\nCluster {cluster_id}: {len(cluster_data)} transações')
    print(f'  Quantidade média: {cluster_data["quantidade"].mean():.2f}')
    print(f'  Preço médio: R$ {cluster_data["preco_unitario"].mean():.2f}')
    print(f'  Receita média: R$ {cluster_data["receita_total"].mean():.2f}')
    print(f'  Desconto médio: {cluster_data["desconto_pct"].mean():.2f}%')

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot: Quantidade vs Receita
for cluster_id in range(k_optimal):
    cluster_data = df[df['cluster'] == cluster_id]
    axes[0].scatter(cluster_data['quantidade'], cluster_data['receita_total'], 
                    label=f'Cluster {cluster_id}', alpha=0.6, s=50)
axes[0].set_xlabel('Quantidade')
axes[0].set_ylabel('Receita Total (R$)')
axes[0].set_title('Clustering: Quantidade vs Receita', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter plot: Desconto vs Receita
for cluster_id in range(k_optimal):
    cluster_data = df[df['cluster'] == cluster_id]
    axes[1].scatter(cluster_data['desconto_pct'], cluster_data['receita_total'], 
                    label=f'Cluster {cluster_id}', alpha=0.6, s=50)
axes[1].set_xlabel('Desconto (%)')
axes[1].set_ylabel('Receita Total (R$)')
axes[1].set_title('Clustering: Desconto vs Receita', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Parte 3: Machine Learning - Previsão de Receita

### Random Forest para Previsão de Receita Total

In [ ]:
# Preparação de dados
df_ml = df[['quantidade', 'preco_unitario', 'custo_unitario', 'desconto_pct', 'receita_total']].fillna(df.mean())

# Features e Target
X = df_ml[['quantidade', 'preco_unitario', 'custo_unitario', 'desconto_pct']]
y = df_ml['receita_total']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modelo
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)

# Previsões
y_pred_train = rf_model.predict(X_train)
y_pred_test = rf_model.predict(X_test)

# Avaliação
mse_train = mean_squared_error(y_train, y_pred_train)
mse_test = mean_squared_error(y_test, y_pred_test)
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)

print('\n=== RANDOM FOREST - PREVISÃO DE RECEITA ===')
print(f'MSE (Treino): {mse_train:,.2f}')
print(f'MSE (Teste): {mse_test:,.2f}')
print(f'R² Score (Treino): {r2_train:.4f}')
print(f'R² Score (Teste): {r2_test:.4f}')
print(f'RMSE (Teste): {np.sqrt(mse_test):,.2f}')

# Feature Importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print('\n=== IMPORTÂNCIA DAS FEATURES ===')
print(feature_importance)

# Gráfico
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='steelblue')
plt.xlabel('Importância')
plt.title('Feature Importance - Random Forest', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico de Previsões vs Realidade
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Treino
axes[0].scatter(y_train, y_pred_train, alpha=0.6, s=30)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
axes[0].set_xlabel('Valor Real')
axes[0].set_ylabel('Valor Previsto')
axes[0].set_title(f'Treino (R² = {r2_train:.4f})', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Teste
axes[1].scatter(y_test, y_pred_test, alpha=0.6, s=30, color='orange')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Valor Real')
axes[1].set_ylabel('Valor Previsto')
axes[1].set_title(f'Teste (R² = {r2_test:.4f})', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Conclusões

In [ ]:
# Resumo Final
print('\n=== RESUMO FINAL DA ANÁLISE ===')
print(f'\n1. ESTATÍSTICAS:')
print(f'   - Total de vendas: {len(df):,}')
print(f'   - Receita total: R$ {df["receita_total"].sum():,.2f}')
print(f'   - Lucro estimado: R$ {((df["receita_total"] * (1 - df["desconto_pct"]/100)) - (df["custo_unitario"] * df["quantidade"])).sum():,.2f}')

print(f'\n2. CLUSTERING:')
print(f'   - Clusters identificados: {k_optimal}')
print(f'   - Cluster 0 (alto volume, preço baixo): {len(df[df["cluster"] == 0])} transações')
print(f'   - Cluster 1 (médio volume, preço médio): {len(df[df["cluster"] == 1])} transações')
print(f'   - Cluster 2 (baixo volume, preço alto): {len(df[df["cluster"] == 2])} transações')

print(f'\n3. PREVISÃO:')
print(f'   - Modelo Random Forest com R² = {r2_test:.4f}')
print(f'   - Feature mais importante: {feature_importance.iloc[0]["feature"]}')
print(f'   - RMSE: R$ {np.sqrt(mse_test):,.2f}')

print(f'\n4. RECOMENDAÇÕES:')
print(f'   - Focar em produtos de alto volume (Cluster 0)')
print(f'   - Ajustar estratégia de precificação para Cluster 2')
print(f'   - Usar modelo de previsão para planejamento de estoque')